# Proposal Generation

## Small-Scale Experiments

### Imports


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import s3fs
import zarr

from scipy.spatial import KDTree
from scipy.ndimage import gaussian_laplace

from time import time
from tifffile import imwrite

from aind_exaspim_soma_detection import soma_proposal_generation as spg
from aind_exaspim_soma_detection.utils import img_util, util

%matplotlib inline

IMG_PREFIX = {
    "685221": "exaSPIM_685221_2024-04-12_11-46-38_fusion_2024-07-22_21-00-15",
    "706301": "exaSPIM_706301_2024-04-23_11-24-24_fusion_2024-05-21_00-00-03",
    "708369": "exaSPIM_708369_2024-04-08_15-20-36_fusion_2024-05-20_23-30-43",
    "709393": "exaSPIM_709393_2024-04-17_09-37-51_fusion_2024-07-27_00-20-20",
    "715345": "exaSPIM_715345_2024-06-07_10-03-37_fusion_2024-07-02_10-30-40"
}

In [2]:
# Filter points
def filter_centers(img_patch, centers, margin):
    # Initializations
    brightness = [img_patch[c] for c in centers]
    if len(centers) > 0:
        kdtree = KDTree(centers)
    else:
        return list(), list()

    # Main
    filtered_centers = list()
    discared_centers = list()
    visited = set()
    for idx in np.argsort(brightness)[::-1]:
        # Determine whether to visit center
        inbounds_bool = spg.is_inbounds(img_patch, centers[idx], d)
        not_visited_bool = centers[idx] not in visited
        if inbounds_bool and not_visited_bool:
            # Get subpatch
            center = tuple([int(v) for v in centers[idx]])
            img_subpatch = img_patch[
                center[0]-margin:center[0]+margin,
                center[1]-margin:center[1]+margin,
                center[2]-margin:center[2]+margin,
            ]

            # Check whether to keep
            center_subpatch = (margin, margin, margin)
            if check_gaussian_fit(img_subpatch, center_subpatch) is not None:
                filtered_centers.append(center)
                discard_nearby_centers(kdtree, visited, center)
            else:
                discared_centers.append(center)
    return filtered_centers, discared_centers


def check_gaussian_fit(img, center, radius=3):
    img_vals, coords, params = spg.fit_gaussian(img, center, radius)
    if params is not None:
        corr_coefficient = spg.fitness_quality(img_vals, coords, params)
        if corr_coefficient < 0.6:
            params = None
    return params


def discard_nearby_centers(kdtree, visited, center):
    idxs = kdtree.query_ball_point(center, 5)
    for voxel in kdtree.data[idxs]:
        visited.add(tuple([int(v) for v in voxel]))


def global_filtering(points):
    kdtree = KDTree(points)
    points = set(points)
    print("# pairs:", len(kdtree.query_pairs(5)))
    for i, j in kdtree.query_pairs(5):
        # Extract Points
        xyz_i = kdtree.data[i]
        xyz_j = kdtree.data[j]
        xyz = tuple([(xyz_i[n] + xyz_j[n]) / 2 for n in range(3)])

        # Update points
        points.discard(tuple([int(x) for x in xyz_i]))
        points.discard(tuple([int(x) for x in xyz_j]))
        points.add(tuple([int(x) for x in xyz]))
    return list(points)


# utils
def get_img_patch(img, voxel, shape, from_center=True):
    start, end = img_util.get_start_end(voxel, shape, from_center=from_center)
    return img[0, 0, start[2]:end[2], start[1]:end[1], start[0]:end[0]]


### Opem img

In [3]:
# Parameters
s3_bucket = "aind-open-data"
dataset = "706301"
downsample_factor = 4

# Initializations
fs = s3fs.S3FileSystem()
s3_url = f"s3://{s3_bucket}/{IMG_PREFIX[dataset]}/fused.zarr/{downsample_factor}/"

# Open img
store = s3fs.S3Map(root=s3_url, s3=fs)
img = zarr.open(store, mode='r')
print("img.shape:", img.shape)

PathNotFoundError: nothing found at path ''

### Read img

In [ ]:
traceable_somas = [
    [18403.324, 11245.411, 2675.9985],
    [23425.596, 8926.942, 2185.401],
    [18400.334,11231.872,2683.3584],
    [21624.133,7740.173,12768.359],
    [22883.316,4323.8574,16284.648],
    [43937.27,5479.154,12344.152],
]

bright_nonsomas = [
    [20026.29,6470.747,14384.427],
    [18330.354,3822.579,14477.964],
    [22800.148,2251.1611,16850.379],
    [37846.895,6913.5713,20431.992],
    [43469.51,4820.5303,16769.32],
    [44225.082,4402.9487,15589.198],
    [22086.203,16967.01,14625.586],
    [20239.145,21295.611,11517.228],
]

In [ ]:
# Region of interest
shape = [128, 128, 128]
from_center = True

# Read img
voxel = img_util.to_voxels([28454.617,6914.559,24268.78], downsample_factor=downsample_factor)
img_patch = get_img_patch(img, voxel, shape, from_center=False)
value = np.max(img_patch)
print("Voxel:", voxel)


In [ ]:
# Parameters
d = 16  # context buffer

# Find blob centroids
t0 = time()
blobs = spg.detect_blobs(img_patch, bright_threshold=60)
centers = spg.get_centroids(img_patch, blobs)

# Heuristic-based Filtering
centers = global_filtering(centers)
centers, discarded_centers = filter_centers(img_patch, centers, d)
t1 = time()

# Visualize
spots_img = np.zeros(img_patch.shape)
for center in centers:
    spots_img = img_util.mark_voxel(spots_img, center)

rejects_img = np.zeros(img_patch.shape)
for center in discarded_centers:
    rejects_img = img_util.mark_voxel(rejects_img, center)

# Plot mips
print("# Objects Detected:", len(centers))
print("Runtime:", t1 - t0)
img_util.plot_mips(img_patch[d:-d, d:-d, d:-d], clip_bool=True)
img_util.plot_mips(spots_img[d:-d, d:-d, d:-d])
img_util.plot_mips(gaussian_laplace(img_patch[d:-d, d:-d, d:-d], 2))


In [ ]:
imwrite("img_patch.tiff", img_patch[d:-d, d:-d, d:-d])
imwrite("proposals.tiff", spots_img[d:-d, d:-d, d:-d])
imwrite("rejects.tiff", rejects_img[d:-d, d:-d, d:-d])

In [ ]:
stop

## Large-Scale Experiments

In [ ]:
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from time import time
from tqdm import tqdm

import os
import random


def generate_proposals(img, offset, window_size):
    # Read patch
    img_patch = get_img_patch(img, offset, window_size, from_center=False)
    if np.max(img_patch) < 60:
        return list(), list()

    # Detect candidates
    blobs = spg.detect_blobs(img_patch, bright_threshold=60)
    centers = spg.get_centroids(img_patch, blobs)
    centers, rejects = filter_centers(img_patch, centers, d)

    # Convert coordinates
    centers = [local_to_physical(voxel[::-1], offset) for voxel in centers]
    rejects = [local_to_physical(voxel[::-1], offset) for voxel in rejects]
    return centers, rejects


def local_to_physical(local_voxel, offset):
    global_voxel = np.array([v + o for v, o in zip(local_voxel, offset)])
    return img_util.to_world(global_voxel * 2 ** downsample_factor)



### Soma Candidate Detection

In [ ]:
# Parameters
window_size = (128, 128, 128)
overlap = (36, 36, 36)

# Initializations
output_dir = "/root/capsule/results/soma_candidates"
util.mkdir(output_dir, delete=True)
t0 = time()

# Detection
offsets = img_util.sliding_window_coords_3d(img, window_size, overlap)
print("# img patches:", len(offsets))
with ThreadPoolExecutor() as executor:
    # Assign threads
    threads = list()
    for offset in offsets: #random.sample(offsets, 100):
        threads.append(
            executor.submit(generate_proposals, img, offset, window_size)
        )

    # Process results
    proposals = list()
    rejects = list()
    pbar = tqdm(total=len(threads))
    for thread in as_completed(threads):
        proposals_i, rejects_i = thread.result()
        proposals.extend(proposals_i)
        #rejects.extend(rejects_i)
        pbar.update(1)
print("# Proposals Generated - before filter:", len(proposals))

# Global Filtering
proposals = global_filtering(proposals)
print("# Proposals Generated - after filter:", len(proposals))


print("Runtime:", time() - t0)

## Save Results

In [ ]:
from random import sample
import boto3

def write_points(output_dir, points, color=None, prefix=""):
    util.mkdir(output_dir, delete=True)
    with ThreadPoolExecutor() as executor:
        # Assign Threads
        threads = list()
        for i, xyz in enumerate(points):
            filename = prefix + str(i + 1) + ".swc"
            path = os.path.join(output_dir, filename)
            threads.append(
                executor.submit(util.save_point, path, xyz, 10, color=color)
            )

def write_to_s3(local_path, bucket_name, prefix):
    """
    Writes a single file on local machine to an s3 bucket.

    Parameters
    ----------
    local_path : str
        Path to file to be written to s3.
    bucket_name : str
        Name of s3 bucket.
    prefix : str
        Path within s3 bucket.

    Returns
    -------
    None

    """
    s3 = boto3.client('s3')
    s3.upload_file(local_path, bucket_name, prefix)


In [ ]:
write_points(proposals, color="0.0 0.0 1.0", prefix="proposal_")
#util.write_points(rejects, color="1.0 0.0 0.0", prefix="reject_")

In [ ]:
local_path = "/root/capsule/data/somas-706301.zip"
bucket_name = "aind-msma-morphology-data"
prefix = "soma-experiments/706301-11262024/somas-706301.zip"
write_to_s3(local_path, bucket_name, prefix)

In [ ]:
write_points( "/root/capsule/results/raw_training_data/706301", sample(proposals, 200), color="0.0 0.0 1.0", prefix="proposal_")

### Test Gaussian fit filtering

In [ ]:
# import numpy as np
import matplotlib.pyplot as plt
from random import sample
from copy import deepcopy


if __name__ == "__main__":
    # Parameters
    d = 12

    # Get valid center
    while True:
        center = sample(centers, 1)[0]
        if spg.is_inbounds(img_patch, center, d=d):
            img_subpatch = get_subpatch(img_patch, center, d)
            break

    # Fit a Gaussian to the blob
    result = check_gaussian_fit(img_subpatch, (d, d, d))

    # Visualization
    fig, axs = plt.subplots(1, 2, figsize=(13, 5))
    axs[0].imshow(img_subpatch[:, :, d], cmap='viridis', origin='lower')
    axs[0].set_title("Original Image (Z slice)")
    if result is not None:
        # Generate the fitted Gaussian for visualization
        x = np.linspace(0, 2 * d, 2 * d)
        y = np.linspace(0, 2 * d, 2 * d)
        z = np.linspace(0, 2 * d, 2 * d)
        x, y, z = np.meshgrid(x, y, z, indexing='ij')
        fitted_blob = spg.gaussian_3d((x, y, z), *result).reshape(x.shape)

        axs[1].imshow(fitted_blob[:, :, d], cmap='viridis', origin='lower')
        axs[1].set_title("Fitted Gaussian (Z slice)")

        plt.tight_layout()
        plt.show()
